In [ ]:
import pandas as pd
import numpy as np
import time
import requests, random
import codecs


from tqdm import tqdm
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains

In [ ]:
pd.set_option('max_columns', None)
pd.set_option('max_rows', None)

In [ ]:
links = pd.read_csv('links.csv',index_col = 0)
links = links.to_numpy()
# links = np.delete(links,[11,16],0)

In [ ]:
len(links)

21

In [ ]:
links

array([['https://ekantipur.com/news'],
       ['https://ekantipur.com/business'],
       ['https://ekantipur.com/opinion'],
       ['https://ekantipur.com/sports'],
       ['https://ekantipur.com/national'],
       ['https://ekantipur.com/koseli'],
       ['https://ekantipur.com/world'],
       ['https://ekantipur.com/entertainment'],
       ['https://ekantipur.com/blog'],
       ['https://ekantipur.com/diaspora'],
       ['https://ekantipur.com/feature'],
       ['https://ekantipur.com/lifestyle'],
       ['https://ekantipur.com/literature'],
       ['https://ekantipur.com/technology'],
       ['https://ekantipur.com/health'],
       ['https://ekantipur.com/pathakmanch'],
       ['https://ekantipur.com/Interview'],
       ['https://ekantipur.com/Art'],
       ['https://ekantipur.com/Other'],
       ['https://ekantipur.com/DASHAIN-ESPICAL'],
       ['https://ekantipur.com/market']], dtype=object)

url = links[2][0]
url

In [ ]:
# Function to perform manual scrolling
def manual_scroll(driver, ht):
    # actions = ActionChains(driver)
    # actions.send_keys(Keys.PAGE_UP).perform()
    int_element = [50,100,500,75,60,80,90,1200,200]
    random_choice = random.choice(int_element)
    driver.execute_script(f"window.scrollTo(0, {ht-random_choice});")
    time.sleep(1)  # Adjust the waiting time as needed

In [ ]:
def extract_infinite_scroll_tags(url, num_scrolls):
    # Initialize the Selenium WebDriver (in this example, using Chrome)
    driver = webdriver.Chrome()

    # Open the webpage
    driver.get(url)

    # Wait for the page to load (adjust the time.sleep value as needed)
    # time.sleep(2)

    try:
        # Perform infinite scrolling for the specified number of scrolls
        last_position = None  # Initialize the last scroll position variable
        count = 0
        for _ in tqdm(range(num_scrolls)):
            # Scroll down to the bottom of the page
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")

            # Wait for the newly loaded content to appear (adjust the time.sleep value as needed)
            time.sleep(2.5)

            # Get the current scroll position after scrolling downward
            current_position = driver.execute_script("return window.pageYOffset;")

            # Check if the scroll position changed
            if current_position == last_position and count < 10:
                count+=1
                # The page cannot scroll downward anymore, break the loop
                manual_scroll(driver, current_position)
                # break
            elif current_position == last_position and count >= 10:
                count = 0
                break

            # Update the last scroll position for the next iteration
            last_position = current_position

        # Get the page source after all scrolls are done
        page_source = driver.page_source

    finally:
        # Close the browser window when done
        driver.quit()

    # Parse the page source using BeautifulSoup
    soup = BeautifulSoup(page_source, 'html.parser')

    # soup = soup.find('section', {"class": "listLayout"})
    # Find all elements with a specific tag name (you can customize this for your use case)
    tags = soup.find_all('article')

    # Return the list of tag elements
    return tags

tags_list[5]

In [ ]:
def create_df(tag_list):
    sites = []
    title = []
    summary = []
    for idx, tag in enumerate(tags_list):
        if tag.find('a')['href'][:5]!='https':
            sites.append('https://ekantipur.com'+tag.find('a')['href'])
        else:
            sites.append(tag.find('a')['href'])
        try:
            title.append(tag.find('a').text)
        except:
            title.append(np.NaN)
        try:
            summary.append(tag.find('p').text)
        except:
            summary.append(np.NaN)

        df = pd.DataFrame({'links':sites,
                           'title':title,
                           'summary':summary
                          })
    return df.dropna()

df = create_df(tags_list)

df.shape

df

df.to_csv('news.csv')

url

___

In [ ]:
links[8:,:]

array([['https://ekantipur.com/blog'],
       ['https://ekantipur.com/diaspora'],
       ['https://ekantipur.com/feature'],
       ['https://ekantipur.com/lifestyle'],
       ['https://ekantipur.com/literature'],
       ['https://ekantipur.com/technology'],
       ['https://ekantipur.com/health'],
       ['https://ekantipur.com/pathakmanch'],
       ['https://ekantipur.com/Interview'],
       ['https://ekantipur.com/Art'],
       ['https://ekantipur.com/Other'],
       ['https://ekantipur.com/DASHAIN-ESPICAL'],
       ['https://ekantipur.com/market']], dtype=object)

url = links[8][0]
url

num_scrolls = 2400
tags_list = extract_infinite_scroll_tags(url, num_scrolls)

len(tags_list)

In [ ]:
for link in links[8:,:].reshape(-1):
    url = link
    num_scrolls = 10000
    tags_list = extract_infinite_scroll_tags(url, num_scrolls)
    print(len(tags_list))
    df = create_df(tags_list)
    file_name = link.split('/')[-1]
    print(file_name)
    df.to_csv(f'{file_name}.csv')
    del tags_list, df

In [ ]:
for link in links[5:7,:].reshape(-1):
    url = link
    num_scrolls = 10000
    tags_list = extract_infinite_scroll_tags(url, num_scrolls)
    print(len(tags_list))
    df = create_df(tags_list)
    file_name = link.split('/')[-1]
    print(file_name)
    df.to_csv(f'{file_name}.csv')
    del tags_list, df

In [ ]:
url = links[0][0]
url

'https://ekantipur.com/news'

In [ ]:
num_scrolls = 20000
tags_list = extract_infinite_scroll_tags(url, num_scrolls)

  0%|                                     | 31/20000 [01:18<14:05:32,  2.54s/it]

In [ ]:
print(len(tags_list))
df = create_df(tags_list)
file_name = url.split('/')[-1]
print(file_name)
df.to_csv(f'{file_name}.csv')
del tags_list, df

15929
news


___

In [ ]:
df = pd.read_csv('news.csv', index_col = 0)

In [ ]:
df.shape

In [ ]:
df.tail()

___

In [ ]:
df['links'].to_numpy()

In [ ]:
url2 = 'https://ekantipur.com/business/2023/07/24/16901824669537677.html'
response = requests.get(url2)

soup2 = BeautifulSoup(response.content, 'html.parser')

In [ ]:
temp_tags = soup2.find('div', {"class":"warp"})

In [ ]:
temp_tags